In [1]:
import os
import gc
import glob
import hashlib

import numpy as np
import pandas as pd
import nibabel as nib
from scipy.ndimage import zoom
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# ---------------- ตั้งค่า ----------------
DATASET_ROOT = os.environ.get("DATASET_ROOT", "/home/jovyan/shared/donut/Alzheimer/OAS1")
SEED = 1334
CHECK_ID = "0454"        # OASIS ใช้ ID 4 หลัก (คนที่ 001 = 0001)
N_RUNS = 5
TARGET_SHAPE = (128, 128, 96)
# -----------------------------------------


In [2]:
df = pd.read_excel(f"{DATASET_ROOT}/_oasis_cross-sectional.xlsx")
df = df[df["CDR"].notna()]
df["label"] = (df["CDR"] > 0).astype(int)

def find_file(subject_id):
    pattern = f"{DATASET_ROOT}/{subject_id}/PROCESSED/MPRAGE/T88_111/*_masked_gfc.hdr"
    files = glob.glob(pattern)
    if len(files) == 0:
        return None
    return files[0]

df["path"] = df["ID"].apply(find_file)
df = df[df["path"].notna()]
df["subject_id"] = df["ID"].apply(lambda x: x.split("_")[1])

print("df หลังกรอง:", df.shape)
print("จำนวน subject ไม่ซ้ำ:", df["subject_id"].nunique())
print(f"{CHECK_ID} อยู่ใน df:", CHECK_ID in set(df["subject_id"]))


df หลังกรอง: (234, 15)
จำนวน subject ไม่ซ้ำ: 234
0454 อยู่ใน df: True


In [3]:
df[df["CDR"].notna()]

,ID,M/F,Hand,Age,Educ,SES,MMSE,CDR,eTIV,nWBV,ASF,Delay,label,path,subject_id
0,OAS1_0001_MR1,F,R,74,2.0,3.0,29.0,0.0,1344,0.743,1.306,NaN,0,/home/jovyan/shared/donut/Alzheimer/OAS1/OAS1_...,0001
2,OAS1_0003_MR1,F,R,73,4.0,3.0,27.0,0.5,1454,0.708,1.207,NaN,1,/home/jovyan/shared/donut/Alzheimer/OAS1/OAS1_...,0003
8,OAS1_0010_MR1,M,R,74,5.0,2.0,30.0,0.0,1636,0.689,1.073,NaN,0,/home/jovyan/shared/donut/Alzheimer/OAS1/OAS1_...,0010
9,OAS1_0011_MR1,F,R,52,3.0,2.0,30.0,0.0,1321,0.827,1.329,NaN,0,/home/jovyan/shared/donut/Alzheimer/OAS1/OAS1_...,0011
11,OAS1_0013_MR1,F,R,81,5.0,2.0,30.0,0.0,1664,0.679,1.055,NaN,0,/home/jovyan/shared/donut/Alzheimer/OAS1/OAS1_...,0013
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
411,OAS1_0453_MR1,F,R,70,1.0,4.0,29.0,0.5,1295,0.748,1.355,NaN,1,/home/jovyan/shared/donut/Alzheimer/OAS1/OAS1_...,0453
412,OAS1_0454_MR1,F,R,73,3.0,2.0,23.0,0.5,1536,0.730,1.142,NaN,1,/home/jovyan/shared/donut/Alzheimer/OAS1/OAS1_...,0454
413,OAS1_0455_MR1,F,R,61,2.0,4.0,28.0,0.0,1354,0.825,1.297,NaN,0,/home/jovyan/shared/donut/Alzheimer/OAS1/OAS1_...,0455
414,OAS1_0456_MR1,M,R,61,5.0,2.0,30.0,0.0,1637,0.780,1.072,NaN,0,/home/jovyan/shared/donut/Alzheimer/OAS1/OAS1_...,0456


In [4]:
def make_split(df, seed):
    # เหมือนเซลล์แบ่งข้อมูลใน Resnet18.ipynb ทุกประการ
    subject_labels = df.groupby("subject_id")["label"].first()
    subject_ids = list(subject_labels.index)
    subject_vals = list(subject_labels.values)

    train_subjects, temp_subjects = train_test_split(
        subject_ids, test_size=0.3, stratify=subject_vals, random_state=seed
    )

    temp_idx_map = dict(zip(subject_ids, subject_vals))
    temp_labels_list = [temp_idx_map[s] for s in temp_subjects]

    val_subjects, test_subjects = train_test_split(
        temp_subjects, test_size=0.5, stratify=temp_labels_list, random_state=seed
    )

    train_df = df[df["subject_id"].isin(train_subjects)]
    val_df = df[df["subject_id"].isin(val_subjects)]
    test_df = df[df["subject_id"].isin(test_subjects)]

    # แก้ assert เดิม: เช็คทีละคู่ (ของเดิมตัดกันสามชุดพร้อมกัน ซึ่งว่างเสมอ)
    tr, va, te = set(train_subjects), set(val_subjects), set(test_subjects)
    assert not (tr & va) and not (tr & te) and not (va & te)

    return train_df, val_df, test_df


In [5]:
def load_and_prep(path):
    img = nib.load(path)
    volume = img.get_fdata()

    if volume.ndim == 4:
        volume = volume[:, :, :, 0]

    nz = np.argwhere(volume > 0)
    lo, hi = nz.min(axis=0), nz.max(axis=0)
    volume = volume[lo[0]:hi[0]+1, lo[1]:hi[1]+1, lo[2]:hi[2]+1]

    p_lo, p_hi = np.percentile(volume, [1, 99])
    volume = np.clip(volume, p_lo, p_hi)
    volume = (volume - p_lo) / (p_hi - p_lo + 1e-8)

    resize_factor = [
        target / current
        for target, current in zip(TARGET_SHAPE, volume.shape)
    ]
    volume = zoom(volume, resize_factor, order=1)
    return volume


def preload(frame, desc):
    # เหมือน preload เดิม แต่คืน subject_id ตามลำดับภาพด้วย เพื่อใช้เป็นหลักฐาน
    images, labels, sids = [], [], []

    for path, lb, sid in tqdm(
        zip(frame["path"].tolist(), frame["label"].tolist(), frame["subject_id"].tolist()),
        total=len(frame), desc=desc, leave=False
    ):
        images.append(load_and_prep(path))
        labels.append(lb)
        sids.append(sid)

    x = np.stack(images).astype(np.float32)[:, None]
    y = np.array(labels)
    return x, y, sids


def checksum(volume):
    return hashlib.sha1(np.ascontiguousarray(volume).tobytes()).hexdigest()[:10]


In [6]:
records = []
split_hashes = []

for run in range(1, N_RUNS + 1):
    print("=" * 70)
    print(f"รอบที่ {run}/{N_RUNS}")

    # ขั้นที่ 1: แบ่ง subject เป็น train / val / test (เริ่มใหม่ทุกรอบ)
    train_df, val_df, test_df = make_split(df, SEED)

    # ขั้นที่ 2: preload ภาพเข้า memory ของทั้งสามชุด
    x_train, y_train, sids_train = preload(train_df, f"รอบ {run} preload train")
    x_val, y_val, sids_val = preload(val_df, f"รอบ {run} preload val")
    x_test, y_test, sids_test = preload(test_df, f"รอบ {run} preload test")

    print(f"  train={len(sids_train)}  val={len(sids_val)}  test={len(sids_test)}")
    print(f"  shape x_train={x_train.shape}")

    # ขั้นที่ 3: หา CHECK_ID ใน 'ข้อมูลที่ preload แล้ว'
    loaded = {
        "train": (sids_train, x_train),
        "val": (sids_val, x_val),
        "test": (sids_test, x_test),
    }
    hits = [(name, sids.index(CHECK_ID), checksum(x[sids.index(CHECK_ID), 0]))
            for name, (sids, x) in loaded.items() if CHECK_ID in sids]

    if len(hits) == 0:
        where, idx, ck = "ไม่พบ", None, None
        print(f"  {CHECK_ID} ไม่อยู่ในชุดใดเลย")
    else:
        assert len(hits) == 1, f"{CHECK_ID} ปรากฏมากกว่า 1 ชุด: {hits}"
        where, idx, ck = hits[0]
        print(f"  {CHECK_ID} -> {where}  (ตำแหน่งที่ {idx} ใน x_{where}, checksum ภาพ {ck})")

    # หลักฐานเสริม: ไม่มี subject ซ้ำข้ามชุดในข้อมูลที่ preload แล้ว
    s_tr, s_va, s_te = set(sids_train), set(sids_val), set(sids_test)
    print(f"  ซ้ำข้ามชุด: train∩val={len(s_tr & s_va)} train∩test={len(s_tr & s_te)} val∩test={len(s_va & s_te)}")

    split_hashes.append(hashlib.sha1(
        ("|".join([",".join(sorted(s_tr)), ",".join(sorted(s_va)), ",".join(sorted(s_te))])).encode()
    ).hexdigest()[:10])

    records.append({
        "run": run, "where": where, "index": idx, "image_checksum": ck,
        "n_train": len(sids_train), "n_val": len(sids_val), "n_test": len(sids_test),
        "split_hash": split_hashes[-1],
    })

    del x_train, x_val, x_test, loaded
    gc.collect()


รอบที่ 1/5


  train=163  val=35  test=36
  shape x_train=(163, 1, 128, 128, 96)
  0454 -> val  (ตำแหน่งที่ 34 ใน x_val, checksum ภาพ 06f99be104)
  ซ้ำข้ามชุด: train∩val=0 train∩test=0 val∩test=0
รอบที่ 2/5


  train=163  val=35  test=36
  shape x_train=(163, 1, 128, 128, 96)
  0454 -> val  (ตำแหน่งที่ 34 ใน x_val, checksum ภาพ 06f99be104)
  ซ้ำข้ามชุด: train∩val=0 train∩test=0 val∩test=0
รอบที่ 3/5


  train=163  val=35  test=36
  shape x_train=(163, 1, 128, 128, 96)
  0454 -> val  (ตำแหน่งที่ 34 ใน x_val, checksum ภาพ 06f99be104)
  ซ้ำข้ามชุด: train∩val=0 train∩test=0 val∩test=0
รอบที่ 4/5


  train=163  val=35  test=36
  shape x_train=(163, 1, 128, 128, 96)
  0454 -> val  (ตำแหน่งที่ 34 ใน x_val, checksum ภาพ 06f99be104)
  ซ้ำข้ามชุด: train∩val=0 train∩test=0 val∩test=0
รอบที่ 5/5


  train=163  val=35  test=36
  shape x_train=(163, 1, 128, 128, 96)
  0454 -> val  (ตำแหน่งที่ 34 ใน x_val, checksum ภาพ 06f99be104)
  ซ้ำข้ามชุด: train∩val=0 train∩test=0 val∩test=0


In [7]:
summary = pd.DataFrame(records)
print(summary.to_string(index=False))
print()
print(f"{CHECK_ID} อยู่ชุดเดียวกันทุกรอบ : {summary['where'].nunique() == 1}  ({', '.join(summary['where'].unique())})")
print(f"ภาพของ {CHECK_ID} เหมือนเดิมทุกรอบ : {summary['image_checksum'].nunique() == 1}")
print(f"รายชื่อ train/val/test เหมือนกันทุกรอบ : {len(set(split_hashes)) == 1}")


 run where  index image_checksum  n_train  n_val  n_test split_hash
   1   val     34     06f99be104      163     35      36 67c0d18d36
   2   val     34     06f99be104      163     35      36 67c0d18d36
   3   val     34     06f99be104      163     35      36 67c0d18d36
   4   val     34     06f99be104      163     35      36 67c0d18d36
   5   val     34     06f99be104      163     35      36 67c0d18d36

0454 อยู่ชุดเดียวกันทุกรอบ : True  (val)
ภาพของ 0454 เหมือนเดิมทุกรอบ : True
รายชื่อ train/val/test เหมือนกันทุกรอบ : True


In [8]:
import json

REF = "split_reference.json"

train_df, val_df, test_df = make_split(df, SEED)
current = {
    "train": sorted(train_df["subject_id"]),
    "val": sorted(val_df["subject_id"]),
    "test": sorted(test_df["subject_id"]),
}

for name, ids in current.items():
    if CHECK_ID in ids:
        print(f"{CHECK_ID} อยู่ใน: {name}")

if not os.path.exists(REF):
    with open(REF, "w") as f:
        json.dump(current, f, indent=2)
    print(f"บันทึกการแบ่งครั้งแรกลง {REF} แล้ว -> Restart kernel แล้วรันใหม่เพื่อเทียบ")
else:
    with open(REF) as f:
        ref = json.load(f)
    for name in ["train", "val", "test"]:
        diff = set(ref[name]) ^ set(current[name])
        if not diff:
            print(f"{name:5s}: เหมือนครั้งแรกทุกคน ({len(current[name])} คน)")
        else:
            print(f"{name:5s}: ต่างจากครั้งแรก {len(diff)} คน เช่น {sorted(diff)[:5]}")

0454 อยู่ใน: val
บันทึกการแบ่งครั้งแรกลง split_reference.json แล้ว -> Restart kernel แล้วรันใหม่เพื่อเทียบ
